# Description: Compute Kappa and Rho for Global Signal

In this notebook we estimate the kappa and the rho of the global signal

In [1]:
import os.path as osp
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import datetime
from utils.basics import PRJ_DIR, PRCS_DATA_DIR, CODE_DIR
from utils.basics import get_dataset_index

In [2]:
import getpass
username = getpass.getuser()
print(username)

javiergc


In [3]:
DATASET = 'evaluation'

In [4]:
ds_index = get_dataset_index(DATASET)
ses_list = list(ds_index.get_level_values('Session').unique())
sbj_list = list(ds_index.get_level_values('Subject').unique())

++ Number of scans    = 439
++ Number of subjects = 221


# 1. Create batch jobs for computing kappa and rho for all scans

We will use the ```generate_metrics``` function within ```tedana``` to compute the kappa and the rho

## 1.1. Create the path to the swarm file

In [5]:
script_path = osp.join(PRJ_DIR,f'swarm.{username}',f'N07_Compute_GS_kappa_and_rho.SWARM.sh')
print(script_path)

/data/SFIMJGC_HCP7T/BCBL2024/swarm.javiergc/N07_Compute_GS_kappa_and_rho.SWARM.sh


## 1.2. Create a folder for log files

In [6]:
log_path = osp.join(PRJ_DIR,f'logs.{username}',f'N07_Compute_GS_kappa_and_rho.log')
if not osp.exists(log_path):
    os.makedirs(log_path)
print(log_path)

/data/SFIMJGC_HCP7T/BCBL2024/logs.javiergc/N07_Compute_GS_kappa_and_rho.log


## 1.3. Write the swarm file

This file will contain one line per scan and calls the program ```GS_kappa_and_rho.py```.

In [7]:
with open(script_path, 'w') as the_file:
    the_file.write('# Script Creation Date: %s\n' % str(datetime.date.today()))
    the_file.write(f'# swarm -f {script_path} -g 8 -t 8 -b 20 --time 00:10:00 --logdir {log_path} --partition quick,norm --module afni \n')
    the_file.write('\n')
    for sbj,ses in tqdm(ds_index):
        the_file.write(f'export SBJ={sbj} RUN={ses} CENSOR_MODE=ALL; cd {CODE_DIR}/python/; sh {CODE_DIR}/python/GS_kappa_and_rho.sh \n')
the_file.close()     

100%|██████████| 439/439 [00:00<00:00, 404948.20it/s]


***
# 2. Check all jobs finalized correctly

As we do that, we will also create a .csv file ```evaluation_gs_kappa_rho.csv``` with the estimated kappa and rho for all scans. 

> **NOTE:** If you see any warnings, that means that the associated batch job did not finish correctly. Try again.

We will be able to load that in other notebooks.

In [8]:
kappa_rho_df = pd.DataFrame(index=ds_index,columns=['kappa (GS)','rho (GS)','kappa_rho_color'])
for sbj,ses in tqdm(ds_index, desc='Scans:'):
    path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'{sbj}_{ses}_GS_kappa_and_rho.ALL.txt')
    if not osp.exists(path):
        print("++ WARNING: GS Kappa/Rho file missing: %s" % path)
        continue
    gs = pd.read_csv(path)
    kappa_rho_df.loc[(sbj,ses),'kappa (GS)'] = gs['kappa'][0]
    kappa_rho_df.loc[(sbj,ses),'rho (GS)'] = gs['rho'][0]
    if gs['kappa'][0] > gs['rho'][0]:
        kappa_rho_df.loc[(sbj,ses),'kappa_rho_color'] = 'lightgreen'
    else:
        kappa_rho_df.loc[(sbj,ses),'kappa_rho_color'] = 'red'
kappa_rho_df = kappa_rho_df.infer_objects()
kappa_rho_df.to_csv(f'./summary_files/{DATASET}_gs_kappa_rho.csv')

Scans:: 100%|██████████| 439/439 [00:01<00:00, 383.38it/s]


***
# Extra Code: used to test the intial version of ```GS_kappa_and_rho.sh```

In [8]:
from tedana.metrics.collect import generate_metrics
from tedana.io import OutputGenerator
import numpy as np
import pandas as pd
import nibabel as nib
import os.path as osp
from utils.basics import PRCS_DATA_DIR, TES_MSEC, PRJ_DIR
import hvplot.pandas
from scipy.stats import zscore
from tqdm import tqdm

In [11]:
tes = list(TES_MSEC['evaluation'].values())
ne  = len(tes)
sbj,ses='sub-180','ses-1'

In [12]:
# Load the adaptive mask
mask_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off','tedana_fastica','adaptive_mask.nii.gz')
mask_img  = nib.load(mask_path)
mask_data = mask_img.get_fdata()
nx,ny,nz  = mask_data.shape
mask_vec  = mask_data.reshape(nx*ny*nz,).astype(int)

In [20]:
# Extract number of acquisitions from first echo
e1_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'pb03.{sbj}.r01.e01.volreg+tlrc.HEAD')
e1_img  = nib.load(e1_path)
e1_data = e1_img.get_fdata()
_,_,_,nt = e1_data.shape

In [49]:
if CENSORING_MODE != 'ALL':
    censor_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'censor_{sbj}_combined_2.1D')
    censor = np.loadtxt(censor_path).astype(bool)
else:
    censor = np.ones(nt).astype(bool)
print("++ Number of censored timepoints = %d of %d available" % (nt-censor.sum(),nt))

++ Number of censored timepoints = 0 of 201 available


In [50]:
data_cat = np.zeros((nx*ny*nz,ne,nt))
for e,ee in enumerate(tqdm(list(TES_MSEC[DATASET].keys()))):
    path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'pb03.{sbj}.r01.{ee}.volreg+tlrc.HEAD')
    img  = nib.load(path)
    data = img.get_fdata()
    data_cat[:,e,:] = data.reshape(nx*ny*nz,nt)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:08<00:00,  2.98s/it]


In [51]:
print("++ INFO: Loading GS Timeseries...")
gs_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off', f'pb06.{sbj}.r01.tedana_fastica_OC.GS.demean.1D') #f'pb03.{opts.sbj}.r01.e02.volreg.GS.demean.1D')
print(' +       GS Path = %s' % gs_path)
if osp.exists(gs_path):
   gs      = zscore(np.loadtxt(gs_path)).reshape(nt,1)
else:
   print('        Error: GS file [%s] not available' % gs_path)
   #return -1
print('+        GS shape is %s' % str(gs.shape))

++ INFO: Loading GS Timeseries...
 +       GS Path = /data/SFIMJGC_HCP7T/BCBL2024/prcs_data/sub-180/D03_Preproc_ses-1_NORDIC-off/pb06.sub-180.r01.tedana_fastica_OC.GS.demean.1D
+        GS shape is (201, 1)


In [52]:
# Load OC Dataset
print("++ INFO: Loading optimally combined dataset...")
oc_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off','tedana_fastica','ts_OC.nii.gz')
oc_img  = nib.load(oc_path)
oc_data = oc_img.get_fdata()
data_optcom = oc_data.reshape(nx*ny*nz,nt)
print(" +       data_optcom.shape=%s" % str(data_optcom.shape))

# Load the Optimally combined data
#oc_path = osp.join(PRCS_DATA_DIR,sbj,f'D02_Preproc_fMRI_{ses}','tedana_r01','ts_OC.nii.gz')
#oc_img  = nib.load(oc_path)
#oc_data = oc_img.get_fdata()
#data_optcom = oc_data.reshape(nx*ny*nz,nt)

++ INFO: Loading optimally combined dataset...
 +       data_optcom.shape=(311296, 201)


In [53]:
# Create Output Generator Object that will write nothing
io_generator = OutputGenerator(reference_img=mask_img,
        convention='orig',
        out_dir=osp.join(PRJ_DIR,'temp'),
        prefix=f'temp.{sbj}.{ses}',
        config="auto",
        overwrite=True,
        make_figures=False,
        verbose=False)

In [54]:
data_optcom[:,censor].shape
gs[censor,:].shape
data_cat.shape

(311296, 3, 201)

In [55]:
component_table, mixing = generate_metrics(data_cat=data_cat[:,:,censor],
                 data_optcom=data_optcom[:,censor], 
                 mixing=gs[censor,:],
                 adaptive_mask=mask_vec,
                 tes=tes,
                 io_generator=io_generator,
                 label='GS',
                 metrics=['kappa','rho'])

In [56]:
component_table

,Component,kappa,rho,optimal sign
0,GS_0,18.121075,60.497775,1
